### 연습문제
- Doc2Vec 라이브러리 이용한 감정분석
- 데이터는 rationgs_train.txt 파일을 로드
    - 특수 문자, 2칸 이상의 공백의 문자를 제거하는 정규화 함수
    - document 컬럼의 데이터에서 중복 데이터를 제거
    - 빈 텍스트, " "가 존재한다면 해당 행 데이터도 제거
    - 상위의 5000개 정도 데이터를 이용
- 토큰화 함수 komoran를 이용
    - 필요한 품사 : NNP, NNG, VV, VA, MAG, XR만을 사용
    - 불용어 단어 : 하다, 되다. 이다, 것, 수, 거
- 데이터에서 독립(document), 종속(label) 변수로 데이터를 나누고 train, test 데이터셋을 나눠준다. 비율은 8:2
- Doc2Vec 객체를 생성하여 학습
    - 매개변수
        - vector_size = 200
        - window = 5
        - min_count = 2
        - dm = 1
        - negative = 5
        - seed = 42
        - epochs = 50
    - 학습 시키는 데이터는 X_train
- X_train, X_test -> 문자열 데이터 -> infer_vector() 함수를 이용해서 임베딩
- 고전 머신러닝 분류 모델을 이용하여 임베딩된 데이터를 독립 변수로 X의 데이터들을 종속 변수로 학습하여 예측
    - 정확도를 확인
    - LogisticRegression(max_iter = 2000, radom_state = 42)
    - LinearSVC(random_state = 42)
    - 두개의 모델을 사용하여 정확도가 좋은 모델을 선택


In [248]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import classification_report, accuracy_score
from konlpy.tag import Komoran
from gensim.models.doc2vec import Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import re


In [193]:
data = pd.read_csv("../data/ratings_train.txt", sep="\t")
data.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [194]:
data['document'] = data['document'].fillna('')

In [195]:
# document의 중복과 빈 텍스트, " " 행 제거
data = data.drop_duplicates(subset=['document'])
data = data[data['document'].str.strip() != '']

In [196]:
# 문자에서 불필요한 글자들을 제거 (정규화)
def nomalize(text):
    # 특수문자 제외 , 공백에 대한 처리 
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

data['document'] = [nomalize(text) for text in data['document'].values]

In [197]:
data = data[:5000]

In [198]:
# 형태소 분석 Komoran을 이용하여 토큰화 
komoran = Komoran()

# 특정 품사만 사용 
allow_pos = [ 'NNP', 'NNG', 'VV', 'VA', 'MAG', 'XR' ]
# 불용어 
stop_word = ['하다', '되다', '이다', '것', '수', '거']

def tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            # 길이를 체크하기 전에 동사, 형용사 에는 '다' 붙이기 활용
            if pos in ['VV', 'VA']:
                word += '다'
            if word not in stop_word and len(word) > 1:
                # allow_pos의 포함되어있고
                # stop_word에 포함되어있지 않으며
                # 단어의 길이가 1보다 큰 문자만 활용
                tokens.append(word)
    return tokens

tokenize_docs = [ tokenize(doc) for doc in data['document'].values ]
tokenize_docs

[['더빙', '진짜', '짜증', '나다', '목소리'],
 ['포스터', '초딩', '영화', '오버', '연기', '가볍다'],
 [],
 ['교도소', '이야기', '솔직히', '재미', '없다', '평점', '조정'],
 ['익살', '연기', '돋보이다', '영화', '스파이더맨', '늙다', '보이다', '커스틴 던스트', '너무나'],
 ['걸음마', '떼다', '초등학교', '학년', '영화', '반개', '아깝다'],
 ['원작', '긴장감', '제대로', '살리다'],
 ['반개',
  '아깝다',
  '나오다',
  '이응경',
  '길용우',
  '연기',
  '생활',
  '정말',
  '발로',
  '납치',
  '감금',
  '반복',
  '반복',
  '드라마',
  '가족',
  '없다',
  '연기',
  '못하다',
  '사람',
  '모이다'],
 ['액션', '없다', '재미', '있다', '영화'],
 ['평점', '낮다', '보다', '헐리우드', '화려', '너무', '길들이다', '있다'],
 [],
 ['눈물', '나서다', '죽다', '향수', '자극', '허진호', '감성', '절제', '멜로', '달인'],
 ['울다', '손들다', '횡단보도', '건너다', '뛰쳐나오다', '이범수', '연기', '드럽다'],
 ['담백', '깔끔', '좋다', '신문', '기사', '로만', '보다', '보다', '자꾸', '잊어버리다', '사람'],
 ['취향',
  '존중',
  '진짜',
  '극장',
  '보다',
  '영화',
  '가장',
  '재다',
  '감동',
  '스토리',
  '어거지',
  '감동',
  '어거지'],
 ['매번', '긴장'],
 ['사람',
  '웃기다',
  '바스코',
  '이기다',
  '락스',
  '바비',
  '이기다',
  '아이돌',
  '깔다',
  '그냥',
  '까다',
  '안달',
  '보이다'],
 ['굿바이 레닌', '표절', '이해', '갈수록', '

In [199]:
Y = data['label'].values

In [200]:
tagged = []

for idx , toks in enumerate(tokenize_docs):
    if len(toks) > 0:
        tagged.append(
            TaggedDocument(words=toks, tags=[f"DOC_{idx}"])
        )
    else:
        tagged.append(
            TaggedDocument(words="", tags=[f"DOC_{idx}"])
        )

In [201]:
# tag을 추가한 문서를 이용해서 Doc2Vec 모델에 학습 데이터로 이용
model = Doc2Vec(
    documents= tagged, 
    vector_size= 500, 
    window= 5, 
    min_count = 2,   # 데이터의 개수가 작기 때문, 실제 3-5
    dm = 1,          # PV-DM 방식
    negative = 5,    # 잘못된 단어간의 배치를 사용하여 학습에 이용
    epochs= 50, 
    seed = 42
)

In [202]:
X = np.vstack(
    [model.dv[f"DOC_{idx}"] for idx in range(len(data['document']))]
)
X.shape

(5000, 500)

In [203]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y,
                                                    test_size=0.2, random_state=42, stratify=Y)

In [204]:
lr = LogisticRegression(max_iter=2000, random_state=42)
svc = SVC(random_state=42)

In [205]:
lr.fit(X_train, Y_train)
pred_lr = lr.predict(X_test)

In [206]:
svc.fit(X_train, Y_train)
pred_svc = svc.predict(X_test)

In [207]:
print(classification_report(Y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.73      0.75      0.74       500
           1       0.75      0.73      0.74       500

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000



In [208]:
print(classification_report(Y_test, pred_svc))

              precision    recall  f1-score   support

           0       0.70      0.81      0.75       500
           1       0.77      0.66      0.71       500

    accuracy                           0.73      1000
   macro avg       0.74      0.73      0.73      1000
weighted avg       0.74      0.73      0.73      1000



------

## 강사님 Ver.

In [209]:
# 라이브러리 설치
# !pip install tqdm

In [210]:
# 진행 상황들을 로그에 표시해주는 라이브러리
from tqdm import tqdm

In [211]:
# 데이터 로드
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [212]:
# 150000개의 데이터 중 결측치의 개수가 5개 인것을 확인
# 결측치의 데이터가 매우 작기때문에 제거
# 제거(drop) + 결측치(na) -> dropna()
# dropna() 함수를 Sreies에서도 내장, DataFrame 내장
df2 = df.copy()
df2['document'] = df2['document'].dropna()

In [213]:
# 실수가 많은 코드들
df2['document'] = df['document'].dropna()
df2 = df['document'].dropna()

In [214]:
# 결측치를 제거
df.dropna(inplace=True)

In [215]:
# 텍스트 정규화 함수
def nomalize(text):
    # text 매개변수에 들어오는 데이터?
        # df에 있는 document 컬럼의 vlaues ->> 리뷰 데이터
    # str(text) 사용하는 이유는?
        # 리뷰의 데이터가 문자가 아닌 경우 문자형으로 변경
    text = re.sub(r"[^각-힣0-9a-zA-Z\s\.]", " ", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [216]:
# df['document']에서 nomalize 함수를 이용
# # nomalize에 들어가는 입자를문자열이 기본
# nomalize(df['document']) 잘못된 부분
# apply() 함수 -> 데ㅇ터프레이에서 사용하는 함수
# ap() 함수를 비롯한 기능 각각의 원소들을 추출하여 어떤 작업(함수)들을 
# df.head().apply(lambda x : str(x), axis=1)
# df.head().map(lambda x : print(X))
df = df.applymap(nomalize)

C:\Users\abohv\AppData\Local\Temp\ipykernel_22468\636701083.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(nomalize)


In [217]:
# 빈 / 공백 텍스트 존쟇ㄹ 때
df.loc[
    df['document'].isin(["", " "])
]

,id,document,label
972,7425748,,0
1840,7095375,,1
2159,7070900,,1
2504,7449459,,0
2648,423224,,1
...,...,...,...
148560,7903625,,0
148940,6907774,,0
149364,8014701,,1
149398,7432786,,0


In [218]:
# 빈 텍스트, 공백 텓스트 -> 길이가 1이하
df = df.loc[
    df['document'].str.len() > 1
]

In [219]:
# document에서 중복된 데이터들은 제거
df = df.drop_duplicates('document')

In [220]:
# 토큰화 함수를 생성
# 특정 품사들만 선택
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'XR']
# 불용어
stop_word = ['하다', '되다', '이다', '것', '수','거']

# Komoram 객채를 생성
komoran  = Komoran()

def tokenize_komoran(text):
    tokens = []
    for word, pos in komoran.pos(text):
        # wrod : 단어
        # pos : 품사
        if pos in allow_pos and word not in  stop_word:
            tokens.append(word)  
    return tokens                                                                            

In [221]:
df = df.head(5000)

In [222]:
# tokens화 된 리스트를 데이터프레임에 새로운 컬럼에 추가
tokenize_sentence = [tokenize_komoran(val) for val in df['document'].values]
tokenize_sentence

[['더빙', '진짜', '짜증', '나', '목소리'],
 ['포스터', '초딩', '영화', '오버', '연기'],
 [],
 ['교도소', '이야기', '솔직히', '재미', '없', '평점', '조정'],
 ['익살', '연기', '돋보이', '영화', '스파이더맨', '늙', '보이', '하', '커스틴 던스트', '너무나'],
 ['막', '걸음마', '떼', '초등학교', '학년', '용', '영화', '별', '반개', '아깝'],
 ['원작', '긴장감', '제대로', '살리'],
 ['반개',
  '아깝',
  '욕',
  '나오',
  '이응경',
  '길용우',
  '연기',
  '생활',
  '이',
  '정말',
  '발로',
  '납치',
  '감금',
  '반복',
  '반복',
  '이',
  '드라마',
  '족도',
  '없',
  '연기',
  '못하',
  '사람',
  '모이'],
 ['액션', '없', '재미', '있', '안', '영화'],
 ['왜', '평점', '낮', '꽤', '보', '헐리우드', '화려', '너무', '길들이', '있'],
 ['짱', '진짜', '짱'],
 ['볼', '때', '눈물', '나서', '죽', '향수', '자극', '허진호', '감성', '절제', '멜로', '달인'],
 ['울', '손들', '횡단보도', '건너', '때', '뛰쳐나오', '이범수', '연기', '드럽'],
 ['담백', '깔끔', '좋', '신문', '기사', '로만', '보다', '보', '자꾸', '잊어버리', '사람'],
 ['취향',
  '존중',
  '진짜',
  '극장',
  '보',
  '영화',
  '장',
  '노',
  '재',
  '노',
  '감동',
  '스토리',
  '어거지',
  '감동',
  '어거지'],
 ['매번', '긴장'],
 ['참',
  '사람',
  '웃기',
  '바스코',
  '이기',
  '락스',
  '코',
  '까',
  '고',
  '바비',
  '이기'

In [223]:
df['document_tokenize'] = tokenize_sentence

In [224]:
df.head()

,id,document,label,document_tokenize
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0,"[더빙, 진짜, 짜증, 나, 목소리]"
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 볍지 않구나,1,"[포스터, 초딩, 영화, 오버, 연기]"
2,10265843,너무재밓었다그래서보는것을추천한다,0,[]
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0,"[교도소, 이야기, 솔직히, 재미, 없, 평점, 조정]"
4,6483659,사이몬페그의 익살스런 연기 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 던...,1,"[익살, 연기, 돋보이, 영화, 스파이더맨, 늙, 보이, 하, 커스틴 던스트, 너무나]"


In [231]:
# 토큰화된 문서에 Tag를 부착
def tagged_docs(tokenize_data):
    tagged = []
    for d_id, toks in enumerate(tokenize_data):
        # toks의 길이가 0이라면 -> 학습에서 의미없는 문장
        if len(toks) == 0:
            continue
        tagged.append(
            TaggedDocument(
                words=toks, tags=[f"DOC_{d_id}"]
            )
        )
    return tagged

In [232]:
# 데이터셋
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

In [233]:
# Doc2Vec에서 사용할 데이터는 train_df의 document_token 컬럼의 데이터를 이용한다
tagged_train = tagged_docs(train_df['document_tokenize'].values)

In [238]:
len(tagged_train)

3927

In [234]:
# Dov2Vec 객체를 생성하여 학습
model2 = Doc2Vec(
    documents=tagged_train,
    vector_size=200,
    window=5,
    dm=1,
    min_count=2,
    negative=5,
    seed=42,
    epochs=50
)

In [235]:
# 단어 사전의 개수 확인 -> 단어별 임베딩 벡터wv
print("단어 사전의 개수 : ", len(model2.wv))

단어 사전의 개수 :  2785


In [237]:
len(model2.dv)

3927

In [239]:
# 새로운 문장을 model에 infer_vector 함수를 이용하여 임베딩
def infer_vector(model, norm_texts, epochs=50, ):
    # norm_texts : 정규화 처리가 끝난 문서들
    # model : 임베딩 모델
    vecs = []
    
    for text in norm_texts:
        # text : norm_texts의 각 원소들의 대입
        tokens = tokenize_komoran(text)
        # 토큰화된 데이터가 길이가 0인 경우
        if len(tokens) == 0:
            # 비어있는 토큰 데이터는 영벡터 / 평균 벡터로 대체 가능
            # 영벡터로 출력
            vecs.append(np.zeros(model.vector_size, dtype=np.float32))
        else:
            vec = model.infer_vector(tokens, epochs=epochs)
            vecs.append(vec)
    return np.vstack(vecs)

In [242]:
x_train = infer_vector(model2, train_df['document'].values)
y_train = train_df['label'].values

x_test = infer_vector(model2, test_df['document'].values)
y_test = test_df['label'].values

In [243]:
x_train.shape

(4000, 200)

In [245]:
# 로지스틱회귀 모델과 선형 서포트벡터 분류 모델에 학습하고 정확도 체크하는 함수
def eval_clf(clf, x_tr, y_tr, x_te, y_te, model_name):
    # 모델에 학습
    clf.fit(x_tr, y_tr)
    # 모델을 통한 예측
    pred = clf.predict(x_te)
    # 정확도
    acc = accuracy_score(y_te, pred)
    print(f"{model_name}모델의 정확도는 : ", round(acc, 4))
    # class report
    cls_report = classification_report(y_te,pred)
    print(f"{model_name}모델의 report : \n", cls_report)
    

In [246]:
# 모델을 생성
logi = LogisticRegression(max_iter=2000, random_state=42)
eval_clf(logi, x_train, y_train, x_test, y_test, "Logistic")

Logistic모델의 정확도는 :  0.739
Logistic모델의 report : 
               precision    recall  f1-score   support

           0       0.73      0.75      0.74       502
           1       0.74      0.72      0.73       498

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000



In [250]:
svc2 = LinearSVC(random_state=42)
eval_clf(svc2, x_train, y_train , x_test, y_test, 'LinearSVC')

LinearSVC모델의 정확도는 :  0.737
LinearSVC모델의 report : 
               precision    recall  f1-score   support

           0       0.73      0.75      0.74       502
           1       0.74      0.72      0.73       498

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000

